# 01 · The model code, explained

`00_recap` shows *what* the project does. This notebook shows *how* the model code in
`src/sindy_wm/models/` works: each module, each function, what it computes and why. Where a
function does something non-obvious, the notebook redoes the key step by hand and checks that it
gives the same answer as the library.

| Part | Module | What it holds |
|---|---|---|
| 1 | | the big picture: the flow, and the interface all models share |
| 2 | `trajectories` | `to_trajectory`, `derivative_data`: from a Level A table to fitting data |
| 3 | `sindy` | `fit_sindy`: polynomial SINDy, and the STLSQ algorithm behind it |
| 4 | `lanchester` | `LanchesterModel`, `fit_lanchester`: the classical square law |
| 5 | `gated` | `GatedModel`, `fit_gated`: SINDy with every term gated by the opponent |
| 6 | `simulation` | `simulate`, `simulate_to_elimination` |
| 7 | `metrics` | `evaluate`, `summarize_eval`, `evaluate_horizons`, `healing_share` |
| 8 | `runs`, `evaluation.plots` | `save_run`, `plot_predictions` |
| 9 | | cheat sheet: which function answers which question |

It uses the same dataset and train/test split as notebook 04, and runs in under a minute.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import inspect
import json
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysindy as ps

from sindy_wm.evaluation.plots import TEAM_COLORS, plot_predictions
from sindy_wm.io import storage
from sindy_wm.models.gated import GatedModel, fit_gated
from sindy_wm.models.lanchester import LanchesterModel, fit_lanchester
from sindy_wm.models.metrics import evaluate, evaluate_horizons, healing_share, summarize_eval
from sindy_wm.models.runs import save_run
from sindy_wm.models.simulation import simulate, simulate_to_elimination
from sindy_wm.models.sindy import fit_sindy
from sindy_wm.models.trajectories import derivative_data, to_trajectory
from sindy_wm.paths import RAW_DIR
from sindy_wm.states.aggregate import level_a

plt.rcParams["figure.dpi"] = 100
pd.set_option("display.width", 160)

EXPERIMENT = "exp03_10m_random_aggression_stepmul"
DATASET = "10m_attack_a090_s1_v1"
STATE_COLUMNS = ["health_blue", "health_red"]  # Blue first, Red second: every model relies on this order
DOWNSAMPLE = 2
SMOOTH_WINDOW = 9

dataset_dir = RAW_DIR / EXPERIMENT / DATASET
manifest = storage.load_manifest(dataset_dir)
index = storage.load_episode_index(dataset_dir).set_index("episode_idx")
dt = manifest["config"]["env"]["seconds_per_step"]
snaps = storage.load_snapshots(dataset_dir)
states = snaps.groupby("episode_idx").apply(level_a, normalize=True, include_groups=False)
states["t"] = states.index.get_level_values("step") * dt

rng = np.random.default_rng(0)  # the same split as notebooks 00 and 04
episode_ids = rng.permutation(index.index.to_numpy())
n_test = int(round(0.2 * len(episode_ids)))
test_ids = sorted(episode_ids[:n_test].tolist())
train_ids = sorted(episode_ids[n_test:].tolist())
print(f"{DATASET}: {len(index)} episodes, {len(train_ids)} for training, {len(test_ids)} for testing")

## Part 1 · The big picture

```
Level A table (one episode)
    │  trajectories.to_trajectory          trim to first damage, downsample, t from 0
    ▼
trajectories {episode_idx: table}
    │  trajectories.derivative_data        smoothed d/dt, per episode, then stacked
    ▼
fit_sindy / fit_lanchester / fit_gated     →  a model
    │
    ├─ simulation.simulate                 integrate along a real trajectory
    ├─ simulation.simulate_to_elimination  integrate until a side reaches 0.02
    ├─ metrics.*                           score it on test episodes
    ├─ evaluation.plots.plot_predictions   draw it against the data
    └─ runs.save_run                       keep it in runs/
```

**One interface, three model types.** `fit_sindy` returns pysindy's own `ps.SINDy` object.
`LanchesterModel` and `GatedModel` are small classes of our own that copy the five methods the
rest of the code uses. That is why every function after the fitting step works on any of them.

In [ ]:
trajectories = {
    i: to_trajectory(states.loc[i], STATE_COLUMNS, trim_to_engagement=True, downsample=DOWNSAMPLE)
    for i in index.index
}
lanchester = fit_lanchester(trajectories, train_ids, STATE_COLUMNS, dt * DOWNSAMPLE, SMOOTH_WINDOW)
gated = fit_gated(trajectories, train_ids, STATE_COLUMNS, SMOOTH_WINDOW, degree=1, threshold=0.05)
poly2 = fit_sindy(
    trajectories, train_ids, STATE_COLUMNS, dt * DOWNSAMPLE, SMOOTH_WINDOW, degree=2, threshold=0.05
)
models = {"poly2 (ps.SINDy)": poly2, "LanchesterModel": lanchester, "GatedModel": gated}

interface = {
    "predict(x)": "derivatives at states x: the right-hand side of dx/dt = f(x)",
    "simulate(x0, t)": "integrate from x0 over the times t",
    "coefficients()": "array (n_states, n_features) of fitted coefficients",
    "get_feature_names()": "the name of each feature (column of coefficients)",
    "equations()": "the model as readable text, one string per state",
}
pd.DataFrame(
    {name: [hasattr(m, method.split("(")[0]) for method in interface] for name, m in models.items()},
    index=pd.Index([f"{m}: {what}" for m, what in interface.items()], name="method"),
)

## Part 2 · `trajectories`: from a Level A table to fitting data

### 2.1 `to_trajectory(ep, state_columns, trim_to_engagement, downsample)`

Three steps, on one episode:

1. **Trim** (if `trim_to_engagement`): find the first row where the state columns *summed* are
   below their value at row 0 (the first damage), and start one row earlier. The approach phase,
   where nothing happens to health, is dropped.
2. **Downsample**: keep every `downsample`-th row.
3. **Reset time**: subtract the first remaining `t`, so every trajectory starts at `t = 0`.

A small made-up episode makes the rule visible:

In [ ]:
toy = pd.DataFrame(
    {
        "t": np.arange(8) * 0.5,
        "health_blue": [1.0, 1.0, 1.0, 1.0, 0.9, 0.8, 0.7, 0.6],
        "health_red": [1.0, 1.0, 1.0, 0.95, 0.9, 0.85, 0.8, 0.7],
    }
)
print("Original: first damage at row 3 (health_red drops)")
display(toy)
print("to_trajectory(..., trim_to_engagement=True, downsample=1): starts one row earlier, at row 2")
display(to_trajectory(toy, STATE_COLUMNS, trim_to_engagement=True, downsample=1))
print("... and downsample=2: every second row of that")
display(to_trajectory(toy, STATE_COLUMNS, trim_to_engagement=True, downsample=2))

### 2.2 `derivative_data(trajectories, ids, state_columns, smooth_window)`

SINDy and Lanchester are fitted on $\dot x$, which the data does not contain. It is estimated with a
**Savitzky–Golay filter**: around each point, fit a quadratic to the `smooth_window` nearest points
and take *its* derivative. A larger window averages over more shots (smoother, but slower to react).

Three details matter:

- **Each episode is differentiated on its own**, and only then are episodes stacked. Stacking first
  would put the end of one battle next to the start of the next, a huge fake derivative.
- **The states are smoothed too.** The same filter gives the smoothed value of $x$ at each point,
  and `derivative_data` returns that, not the raw staircase. This is what `ps.SINDy` does
  internally, so the library is evaluated on states that match the derivatives.
- `fit_lanchester` and `fit_gated` call `derivative_data` directly; `fit_sindy` lets pysindy do
  the same thing with the same settings. All three models are fitted on identical data
  (a test in `tests/test_sindy_models.py` checks this).

In [ ]:
ep_id = test_ids[0]
traj = trajectories[ep_id]
x, t = traj[STATE_COLUMNS].to_numpy(), traj["t"].to_numpy()

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, ps.FiniteDifference()(x, t=t)[:, 1], color="0.75", lw=0.8, label="plain finite difference")
for window, color in [(5, "tab:orange"), (9, "tab:red"), (17, "tab:purple")]:
    smooth = ps.SmoothedFiniteDifference(smoother_kws={"window_length": window, "polyorder": 2})(x, t=t)
    ax.plot(t, smooth[:, 1], color=color, label=f"Savitzky-Golay, window {window}")
ax.set(title=f"d(health_red)/dt, episode {ep_id}", xlabel="time since first damage (s)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.show()

X, DX = derivative_data(trajectories, train_ids, STATE_COLUMNS, SMOOTH_WINDOW)
print(
    f"derivative_data: X {X.shape} (states), DX {DX.shape} (their derivatives), "
    f"stacked over {len(train_ids)} training episodes"
)

## Part 3 · `sindy`: `fit_sindy` and the STLSQ algorithm

`fit_sindy(trajectories, train_ids, state_columns, dt, smooth_window, degree, threshold)` is a thin
wrapper around pysindy. The idea of SINDy:

$$\dot X \approx \Theta(X)\,\Xi$$

- $\Theta(X)$ is the **library**: one column per candidate term, evaluated at every row. With a
  degree-2 polynomial library and state $(B, R)$: $1, B, R, B^2, BR, R^2$.
- $\Xi$ holds the coefficients, one column per state. **Sparse** means most entries are 0.

**STLSQ** (sequentially thresholded least squares) finds a sparse $\Xi$, one state at a time:

1. Solve a ridge regression on all candidate terms:
   $\min_\xi \lVert \dot x - \Theta\xi \rVert^2 + \alpha \lVert \xi \rVert^2$ (pysindy's default $\alpha = 0.05$).
2. Set every coefficient with $|\xi_k| <$ `threshold` to 0 and drop those terms.
3. Solve again on the remaining terms. Repeat until the set of terms stops changing.
4. Finally refit the kept terms with plain least squares ("unbias").

Below, the algorithm written out on the data from `derivative_data`, compared with `fit_sindy`:

In [ ]:
def stlsq_by_hand(theta, y, threshold, alpha=0.05, max_iter=20):
    active = np.ones(theta.shape[1], dtype=bool)
    rounds = []
    for _ in range(max_iter):
        a = theta[:, active]
        xi = np.zeros(theta.shape[1])
        xi[active] = np.linalg.solve(a.T @ a + alpha * np.eye(a.shape[1]), a.T @ y)  # step 1/3: ridge
        rounds.append(xi.copy())
        new_active = np.abs(xi) >= threshold  # step 2: threshold
        if (new_active == active).all():
            break
        active = new_active
    xi = np.zeros(theta.shape[1])
    xi[active] = np.linalg.lstsq(theta[:, active], y, rcond=None)[0]  # step 4: unbias
    return xi, rounds


B, R = X[:, 0], X[:, 1]  # smoothed states from derivative_data
theta = np.column_stack([np.ones_like(B), B, R, B**2, B * R, R**2])
names = poly2.get_feature_names()
print("pysindy's library:", names)

THRESHOLD = 0.05
for k, state in enumerate(STATE_COLUMNS):
    xi, rounds = stlsq_by_hand(theta, DX[:, k], THRESHOLD)
    print(f"\n({state})'   STLSQ rounds (terms below {THRESHOLD} are dropped after each round):")
    display(
        pd.DataFrame(rounds, columns=names, index=[f"round {r + 1}" for r in range(len(rounds))]).round(3)
    )
    comparison = pd.DataFrame({"by hand": xi, "fit_sindy": poly2.coefficients()[k]}, index=names)
    display(comparison.T.round(4))

The hand-written version and `fit_sindy` agree. Two things worth noticing in the rounds:

- `threshold` acts on the **size of the coefficient**, so it depends on the scale of each feature.
  That is why the same threshold removes fewer terms in the gated library (Part 5), whose features
  are smaller.
- Terms are only ever removed, never added back. A term that is wrongly dropped early stays out.

## Part 4 · `lanchester`: the classical square law

$$\dot B = -a R, \qquad \dot R = -b B$$

`fit_lanchester` does not search a library. Each equation has a single coefficient and no
intercept, so least squares has a closed form: minimizing $\sum (\dot B + aR)^2$ over $a$ gives

$$a = -\frac{\langle R, \dot B\rangle}{\langle R, R\rangle}, \qquad b = -\frac{\langle B, \dot R\rangle}{\langle B, B\rangle}$$

with the same derivatives as everything else. `LanchesterModel` stores `a` and `b` and implements
the shared interface, with $(B, R)$ as its features.

In [ ]:
a_by_hand = -np.dot(R, DX[:, 0]) / np.dot(R, R)
b_by_hand = -np.dot(B, DX[:, 1]) / np.dot(B, B)
print(f"by hand:        a = {a_by_hand:.4f}, b = {b_by_hand:.4f}")
print(f"fit_lanchester: a = {lanchester.a:.4f}, b = {lanchester.b:.4f}")
print("equations:", lanchester.equations())
print("coefficients():\n", lanchester.coefficients(), "  features:", lanchester.get_feature_names())

# Physical check: a marine deals 6 damage per 0.61 s and has 45 hp, so a full team firing would
# drain the other's normalized health at 6 / 0.61 / 45 per second, times the share of shooters left
full_firepower = 6 / 0.61 / 45
print(
    f"\nfull firepower: {full_firepower:.3f}/s -> fitted a, b are {lanchester.a / full_firepower:.0%} "
    f"and {lanchester.b / full_firepower:.0%} of it (moving, out of range, overkill)"
)

## Part 5 · `gated`: SINDy with every term gated by the opponent

### 5.1 The model form

$$\dot B = -R\, p_B(B, R), \qquad \dot R = -B\, p_R(R, B)$$

Each side's losses are the opponent's strength times a polynomial **firepower factor** $p$, a
polynomial in the side's *own* quantity and the opponent's, up to `degree`. Because every term
contains the opponent, a side cannot change once the opponent is at 0: no healing after a battle,
by construction. Degree 0 ($p$ constant) is exactly the Lanchester square law.

`GatedModel.exponents` lists the terms of $p$ as (power of own, power of opp):

In [ ]:
for degree in [0, 1, 2]:
    m = GatedModel(STATE_COLUMNS, degree, np.zeros((2, 1)))
    print(f"degree {degree}: exponents {m.exponents}")
    print(f"          feature names {m.get_feature_names()}")

`features(x, side)` builds the library for one equation: column $k$ is
$-\text{opp} \cdot \text{own}^{i} \cdot \text{opp}^{j}$. The minus sign is built in, so a **positive
coefficient means more losses**. For Blue's equation, own = Blue and opp = Red; for Red's, the other
way round. Three example states, degree 1:

In [ ]:
example = np.array([[1.0, 1.0], [0.6, 0.3], [0.7, 0.0]])  # columns: health_blue, health_red
m = GatedModel(STATE_COLUMNS, 1, np.zeros((2, 3)))
for side, label in [
    (0, "Blue's equation (own = Blue, opp = Red)"),
    (1, "Red's equation (own = Red, opp = Blue)"),
]:
    print(label)
    display(
        pd.DataFrame(
            m.features(example, side),
            columns=m.get_feature_names(),
            index=[f"B={b}, R={r}" for b, r in example],
        )
    )
print("Last row: Red is gone, so every feature of Blue's equation is 0 -> Blue's health stays put.")

### 5.2 `fit_gated`

The same STLSQ as `fit_sindy` (`ps.STLSQ(threshold)`), run once per equation on that equation's
gated features and the derivatives from `derivative_data`. Only the library differs from
`fit_sindy`.

**Check on a known system.** Generate battles from a model whose coefficients we choose, fit it back,
and compare. The starting states are deliberately uneven: from (1, 1) only, $B \approx R$ along the
whole battle and the terms cannot be told apart (the identifiability problem behind the
uneven-sides experiment).

In [ ]:
true = GatedModel(["b", "r"], degree=1, coef=[[0.4, -0.2, 0.0], [0.3, 0.0, 0.0]])
t_toy = np.arange(40) * 0.1
toy_trajectories = {
    k: pd.DataFrame({"t": t_toy, "b": sim[:, 0], "r": sim[:, 1]})
    for k, x0 in enumerate([(1.0, 1.0), (1.0, 0.6), (0.6, 1.0), (0.8, 0.9)])
    for sim in [true.simulate(np.array(x0), t_toy, integrator_kws={"method": "LSODA", "rtol": 1e-9})]
}
recovered = fit_gated(
    toy_trajectories, list(toy_trajectories), ["b", "r"], smooth_window=5, degree=1, threshold=0.05
)
print("true:     ", true.equations())
print("recovered:", recovered.equations())

### 5.3 On the real data

`equations()` writes the model out with the real column names (`_monomial` builds names like
`health_red^2 health_blue`). Read Blue's equation as $-R \cdot (\dots)$: the factor in brackets is
Red's effective firepower per unit of Red health.

In [ ]:
for s, eq in zip(STATE_COLUMNS, gated.equations(), strict=True):
    print(f"({s})' = {eq}")
c = gated.coefficients()
print(f"\nBlue's losses: -R * ({c[0, 0]:.3f} {c[0, 1]:+.3f}*B {c[0, 2]:+.3f}*R)")
print(f"Red's losses:  -B * ({c[1, 0]:.3f} {c[1, 1]:+.3f}*R {c[1, 2]:+.3f}*B)")

The negative terms on the opponent's own health (−R inside Blue's bracket, −B inside Red's) say:
firepower per unit is *lower* when the shooting side is at full health. Recap Part 6 and notebook 04
Part 6 explain why: full health means early in the fight, when many units are still walking into
range. It is a real effect, but an engagement effect in disguise, not a combat law.

## Part 6 · `simulation`: running a model forward

### 6.1 `simulate(model, traj, state_columns, start=0)`

Integrates from the real state at row `start` over the trajectory's own remaining time points
(LSODA, a solver that switches method if the equations get stiff), so the result lines up row for
row with the data. It returns **`None` instead of raising** if the integration fails or blows up, so
one bad model cannot crash an evaluation loop.

In [ ]:
traj = trajectories[test_ids[0]]
sim = simulate(gated, traj, STATE_COLUMNS, start=len(traj) // 2)
print(f"trajectory has {len(traj)} rows; simulating from row {len(traj) // 2} gives {sim.shape[0]} rows")
print(
    pd.DataFrame(
        np.column_stack([traj[STATE_COLUMNS].to_numpy()[len(traj) // 2 :], sim]),
        columns=["blue actual", "red actual", "blue model", "red model"],
    )
    .iloc[::8]
    .round(3)
)

### 6.2 `simulate_to_elimination(model, x0, t_max, threshold=0.02)`

Integrates from `x0` **until a side is eliminated**, without being told when the real battle ended.
It uses the ODE solver's *events*: functions of the state that the solver watches for a sign change.

- `eliminated(t, x) = min(B, R) - threshold` crosses 0 when a side reaches the threshold,
- `blown_up(t, x) = 1000 - max|x|` crosses 0 if the model explodes.

Both are *terminal*, so the solver stops right there and reports the exact crossing time. If
`eliminated` fired, the side with more left has won; otherwise the result is `(None, None)`.

**Check against a closed form.** With $a = b = k$ the square law can be solved exactly:
$R(t) = R_0\cosh(kt) - B_0\sinh(kt)$, so the elimination time is where that reaches 0.02.

In [ ]:
from scipy.optimize import brentq

k, b0, r0 = 0.5, 1.0, 0.6
square_law = LanchesterModel(["b", "r"], a=k, b=k)
exact = brentq(lambda t: r0 * np.cosh(k * t) - b0 * np.sinh(k * t) - 0.02, 0, 20)
blue_won, t_elim = simulate_to_elimination(square_law, np.array([b0, r0]), t_max=60)
print(f"closed form: Red eliminated at t = {exact:.6f} s")
print(f"simulate_to_elimination: blue_won = {blue_won}, t = {t_elim:.6f} s")

mid = len(traj) // 2
for name, m in models.items():
    print(
        f"{name:>17} from midway of episode {test_ids[0]}:",
        simulate_to_elimination(m, traj[STATE_COLUMNS].iloc[mid].to_numpy(), t_max=60),
    )
print(
    f"{'real':>17}: Blue won = {bool(index.loc[test_ids[0], 'battle_won'])}, "
    f"{traj['t'].iloc[-1] - traj['t'].iloc[mid]:.2f} s after midway"
)

## Part 7 · `metrics`: scoring a model

### 7.1 `evaluate` and `summarize_eval`: who wins, and when?

`evaluate` returns one row per test episode, simulating from two starting points: `start` (the first
row) and `midway` (row `len // 2`). For each it records:

| column | meaning |
|---|---|
| `winner_correct_*_elim` | `simulate_to_elimination` picked the real winner (main metric) |
| `t_elim_*`, `duration_actual_*` | predicted and real remaining time (s) |
| `no_elim_*` | nobody was eliminated within 3× the longest test episode (counts as wrong) |
| `rmse_*` | RMSE of `simulate` against the real trajectory |
| `winner_correct_*` | more health left at the *real* end time (uses the real length: reference only) |
| `leader_correct_midway` | baseline: "whoever has more health at midway wins" |

`summarize_eval` averages these into one dictionary, adding `majority_baseline` (always guess the
more common winner) and `n_terms`.

In [ ]:
per_episode = evaluate(gated, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])
display(per_episode.head())
summary = pd.DataFrame(
    {
        name: summarize_eval(m, test_ids, trajectories, STATE_COLUMNS, index["battle_won"])
        for name, m in models.items()
    }
)
summary.round(3)

### 7.2 `evaluate_horizons`: how good is the model a few seconds ahead?

This is how a world model is used. From start points every `every` seconds in each test episode
(after `history` seconds), the model is integrated from the **real** state to each horizon, and the
prediction (clipped at 0) is compared with the real state then. The two baselines see the same start
points:

- `persistence`: nothing changes,
- `last_slope`: the average rate over the last `history` seconds continues.

The figure shows this for one episode and the 2 s horizon: each short line is one prediction.

In [ ]:
ep_id = test_ids[1]
traj = trajectories[ep_id]
x, t = traj[STATE_COLUMNS].to_numpy(), traj["t"].to_numpy()
step = t[1] - t[0]
k_hist, k_every, H = round(1.0 / step), round(1.0 / step), 2.0

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
for ax, (label, predict) in zip(
    axes,
    [
        (
            "GatedModel",
            lambda k0: gated.simulate(x[k0], np.array([0.0, H]), integrator_kws={"method": "LSODA"})[-1],
        ),
        ("last_slope baseline", lambda k0: x[k0] + (x[k0] - x[k0 - k_hist]) / (k_hist * step) * H),
    ],
    strict=True,
):
    for j, col in enumerate(STATE_COLUMNS):
        ax.plot(t, x[:, j], color=TEAM_COLORS[col.split("_")[-1]], alpha=0.5)
    for k0 in range(k_hist, len(x) - round(H / step), k_every):
        end = np.clip(predict(k0), 0, None)
        for j, col in enumerate(STATE_COLUMNS):
            ax.plot([t[k0], t[k0] + H], [x[k0, j], end[j]], color=TEAM_COLORS[col.split("_")[-1]], lw=1.5)
            ax.plot(t[k0], x[k0, j], "k.", ms=4)
    ax.set(title=f"{label}: predictions {H:g} s ahead, episode {ep_id}", xlabel="time since first damage (s)")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("health")
fig.tight_layout()
plt.show()

horizon_tables = {
    name: evaluate_horizons(m, test_ids, trajectories, STATE_COLUMNS, every=1.0) for name, m in models.items()
}
pd.concat(
    {
        **{name: tbl["model"] for name, tbl in horizon_tables.items()},
        "persistence": horizon_tables["GatedModel"]["persistence"],
        "last_slope": horizon_tables["GatedModel"]["last_slope"],
    },
    axis=1,
).round(4)

### 7.3 `healing_share`: does the model ever create health?

`healing_share(model)` evaluates `model.predict` on a 30 × 30 grid over $[0.02, 1]^2$ and returns the
share of grid points where either derivative is positive. The maps show where that happens (red):

In [ ]:
g = np.linspace(0.02, 1.0, 60)
grid = np.array([[b, r] for r in g for b in g])
fig, axes = plt.subplots(1, len(models), figsize=(4.2 * len(models), 3.8), sharey=True)
for ax, (name, m) in zip(axes, models.items(), strict=True):
    heals = (np.asarray(m.predict(grid)) > 1e-9).any(axis=1).reshape(len(g), len(g))
    ax.imshow(heals, origin="lower", extent=(0.02, 1, 0.02, 1), cmap="RdYlGn_r", vmin=0, vmax=1)
    ax.plot([0.02, 1], [0.02, 1], "k:", lw=0.8)
    ax.set(title=f"{name}\nhealing_share = {healing_share(m):.2f}", xlabel="health_blue")
axes[0].set_ylabel("health_red")
fig.tight_layout()
plt.show()

Battles start at (1, 1) and spend most of their time near the dotted diagonal. The polynomial
model heals in the upper left, where Red is far ahead: Red's health would *grow* there. Few test
states lie in that region, so the accuracy numbers above barely show it, but lopsided battles (and
a planner trying out unusual moves) end up exactly there.

## Part 8 · `runs.save_run` and `evaluation.plots.plot_predictions`

`plot_predictions` draws a few episodes: real (solid), simulated from the start (dashed) and from
midway (dotted). `save_run` writes a new, timestamped folder and refuses to overwrite an existing
one. Here into a temporary folder, not `runs/`:

In [ ]:
won = [i for i in test_ids if index.loc[i, "battle_won"]][:2]
lost = [i for i in test_ids if not index.loc[i, "battle_won"]][:2]
fig = plot_predictions(
    gated, won + lost, trajectories, STATE_COLUMNS, index["battle_won"], title="GatedModel, degree 1"
)
plt.show()

run_dir = save_run(
    Path(tempfile.mkdtemp()),
    name="explained_demo",
    config={"dataset": {"name": DATASET}, "model": {"type": "gated", "degree": 1, "threshold": 0.05}},
    model=gated,
    state_columns=STATE_COLUMNS,
    metrics=summarize_eval(gated, test_ids, trajectories, STATE_COLUMNS, index["battle_won"]),
    per_episode=per_episode,
    figures={"test_predictions.png": fig},
)
print(sorted(p.relative_to(run_dir).as_posix() for p in run_dir.rglob("*")))
print(json.dumps(json.loads((run_dir / "model.json").read_text()), indent=1))

## Part 9 · Cheat sheet

| question | function | module |
|---|---|---|
| Turn a Level A episode into fitting data | `to_trajectory` | `trajectories` |
| Get the smoothed derivatives a model is fitted on | `derivative_data` | `trajectories` |
| Fit a general polynomial model | `fit_sindy` | `sindy` |
| Fit the classical square law | `fit_lanchester` | `lanchester` |
| Fit a model that cannot heal after a battle | `fit_gated` | `gated` |
| Run a model along a real episode | `simulate` | `simulation` |
| Who does the model say wins, and when? | `simulate_to_elimination` | `simulation` |
| Score winner and duration predictions | `evaluate`, `summarize_eval` | `metrics` |
| Score predictions a few seconds ahead (world model) | `evaluate_horizons` | `metrics` |
| Is the model physically valid everywhere? | `healing_share` | `metrics` |
| Draw a model against the data | `plot_predictions` | `evaluation.plots` |
| Keep a result | `save_run` | `runs` |

Every function's docstring has the full argument list: `help(fit_gated)`, or hover in VS Code.

In [ ]:
for module in ["trajectories", "sindy", "lanchester", "gated", "simulation", "metrics", "runs"]:
    mod = __import__(f"sindy_wm.models.{module}", fromlist=["_"])
    members = [
        n
        for n, o in inspect.getmembers(mod, lambda o: inspect.isfunction(o) or inspect.isclass(o))
        if not n.startswith("_") and o.__module__ == mod.__name__
    ]
    print(f"sindy_wm.models.{module:<13} {', '.join(members)}")